# Fundamentos de *Deep Learning I* 
> Héctor J. Hortúa, PhD · Instituto de Neurociencias -SavIA-Lab
## Tensores y diferenciación automática con **PyTorch**

> **Herramientas y manejo de datos**
> **Framework:** PyTorch

Este notebook recorre exactamente los mismos conceptos que el de TensorFlow, pero con **PyTorch**. La idea es que veas cómo **las ideas son idénticas** (tensor, rango, *broadcasting*, gradiente) y solo cambia la sintaxis. Si dominas ambos, podrás leer y adaptar código de cualquier repositorio.

### Objetivos de aprendizaje

1. Crear y manipular **tensores de PyTorch**: rango, forma, tipo, indexado, *reshape*.
2. Operar con tensores: elemento a elemento, matricial, reducciones, *broadcasting*.
3. Mover tensores entre **CPU y GPU** y entre **PyTorch y NumPy**.
4. Entender `requires_grad`, `.backward()` y `.grad`: el sistema **autograd**.
5. Calcular derivadas automáticamente, base del entrenamiento de redes.


## Popularidad y filosofía

PyTorch, desarrollado por Meta (Facebook) y liberado en 2016, es el framework **dominante en investigación** y muy fuerte también en producción. Su filosofía es **"Python primero"**: se siente como escribir NumPy con superpoderes (GPU + diferenciación automática).

A diferencia del TensorFlow clásico (que construía un grafo estático antes de correr), PyTorch usa un **grafo dinámico ("define por ejecución")**: el grafo de cómputo se construye sobre la marcha, en cada *forward*. Esto hace la depuración natural —usas `print` y *breakpoints* como en cualquier programa Python— y facilita modelos con estructura variable (muy útil en NLP). Desde TF2, ambos frameworks se parecen mucho; PyTorch simplemente fue así desde el inicio.

## Preparación del entorno

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo disponible:", device)

## Modelo de ejecución: dinámico y *eager* por naturaleza

En PyTorch **todo es eager por defecto**: cada operación se ejecuta al instante, como Python normal. No hay una distinción "eager vs graph" tan marcada como en TF. Cuando necesitas acelerar para producción, existen herramientas de compilación:

- **`torch.compile(modelo)`** (PyTorch 2.x): compila el modelo a un grafo optimizado con una sola línea, sin cambiar tu código.
- **TorchScript / `torch.jit`**: forma más antigua de serializar y optimizar modelos.

La regla práctica es la misma que en TF: **prototipa en eager, compila al final**.

In [ ]:
def polinomio(x):
    return 2 * x**2 + 3 * x + 5

x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print("Resultado:", polinomio(x))
# Para acelerar en producción (PyTorch 2.x):
#poli_rapido = torch.compile(polinomio)

## Tensores y rangos

Igual que en TensorFlow: un **tensor** es un arreglo n-dimensional homogéneo, y el **rango** (aquí se consulta con `.dim()` o `.ndim`) es el número de ejes.

| Rango | Nombre | Ejemplo |
|---|---|---|
| 0 | Escalar | `4` |
| 1 | Vector | `[2, 3, 4]` |
| 2 | Matriz | tabla filas × columnas |
| 3+ | Tensor | imágenes, *batches*, secuencias |

In [ ]:
escalar = torch.tensor(4)                                  # rango 0
vector  = torch.tensor([2.0, 3.0, 4.0])                    # rango 1
matriz  = torch.tensor([[1, 2], [3, 4], [5, 6]], dtype=torch.float32)  # rango 2

print("escalar:", escalar, "| rango:", escalar.dim())
print("vector :", vector,  "| rango:", vector.dim())
print("matriz :\n", matriz, "\n| rango:", matriz.dim())

In [ ]:
rank_3 = torch.tensor([
    [[0, 1, 2, 3, 4],   [5, 6, 7, 8, 9]],
    [[10,11,12,13,14],  [15,16,17,18,19]],
    [[20,21,22,23,24],  [25,26,27,28,29]],
])
print("shape:", rank_3.shape, "| rango:", rank_3.dim())
print(rank_3)

### Indexado y *slicing*

Idéntico a NumPy y a TensorFlow.

In [ ]:
print("Columna 4 de todas las capas:\n", rank_3[:, :, 4])
print("\nSegunda capa completa:\n", rank_3[1, :, :])

### Puente con NumPy

`.numpy()` convierte a arreglo NumPy y `torch.from_numpy(...)` hace lo inverso. **Atención:** en CPU comparten memoria (modificar uno afecta al otro); en GPU primero hay que traer el tensor a CPU con `.cpu()`.

In [ ]:
arr = rank_3.numpy()
print(type(arr), arr.shape)

de_numpy = torch.from_numpy(np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32))
print("Desde NumPy:\n", de_numpy)

## Operaciones sobre tensores

Como en TF, hay operaciones **elemento a elemento** y **matriciales**. El producto de matrices se hace con `torch.matmul` o el operador `@`.

In [ ]:
a = torch.tensor([[1, 2], [3, 4]])
b = torch.ones(2, 2, dtype=torch.long)

print("Suma:\n", a + b)
print("Producto elemento a elemento:\n", a * b)
print("Producto MATRICIAL:\n", a @ b)

### Reducciones

In [ ]:
c = torch.tensor([[4.0, 5.0], [10.0, 1.0]])
print("máximo:", c.max().item())
print("media :", c.mean().item())
print("argmax:", c.argmax().item(), "(índice en el tensor aplanado)")

In [ ]:
d = torch.empty(200).uniform_(-10, 10)
plt.figure(figsize=(7, 3))
plt.plot(torch.relu(d).numpy())
plt.title("ReLU aplicada a 200 valores aleatorios (PyTorch)")
plt.xlabel("índice"); plt.ylabel("relu(x)")
plt.show()

## `torch.where`: condicionales vectorizados

`torch.where` es el equivalente de PyTorch al *if* vectorizado de NumPy. Tiene dos usos:

- **`torch.where(condicion)`** (o `.nonzero()`) devuelve las **posiciones** donde la condición es verdadera.
- **`torch.where(condicion, a, b)`** elige elemento a elemento entre `a` y `b` según la condición: un *if* ternario aplicado a todo el tensor.

Es la herramienta para construir activaciones o máscaras sin escribir bucles.

In [ ]:
x = torch.tensor([1, -2, 3, -4, 5])
idx = (x > 0).nonzero(as_tuple=False)
print("posiciones donde x > 0:", idx.flatten().tolist())   # [0, 2, 4]

### ReLU manual con `where`

In [ ]:
x = torch.tensor([1, -2, 3, -4, 5])
y = torch.where(x > 0, x, torch.zeros_like(x))
print(y)   # [1, 0, 3, 0, 5]  -> reemplaza los negativos por 0

### *If* ternario elemento a elemento

Si la condición se cumple deja `x`; si no, usa `-x`.

In [ ]:
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
cond = x > 3
print(torch.where(cond, x, -x))

## Propiedades de un tensor

- **`.shape`** — forma (longitud de cada eje).
- **`.dim()` / `.ndim`** — número de ejes.
- **`.dtype`** — tipo de dato.
- **`.numel()`** — número total de elementos.
- **`.device`** — CPU o GPU donde vive el tensor (¡específico de PyTorch!).

In [ ]:
t = torch.zeros(3, 2, 4, 5)
print("shape :", t.shape)
print("dim   :", t.dim())
print("dtype :", t.dtype)
print("numel :", t.numel(), "(= 3*2*4*5)")
print("device:", t.device)

## *Reshape*: `view` y `reshape`

PyTorch ofrece dos métodos:

- **`.view(...)`** — no copia datos (requiere memoria contigua); más eficiente.
- **`.reshape(...)`** — copia si hace falta; más flexible.

El `-1` significa *"infiere esta dimensión"*, igual que en TF/NumPy.

In [ ]:
print("Aplanado:", rank_3.reshape(-1))
print("\nDe [3,2,5] a [6,5]:\n", rank_3.reshape(6, 5))
print("\nCon -1  ->  [3,-1] equivale a [3,10]:", rank_3.reshape(3, -1).shape)

## *Broadcasting*

Idéntica regla que en NumPy y TensorFlow: las dimensiones de tamaño 1 se "estiran" virtualmente para hacer compatibles las formas, sin copiar memoria.

In [ ]:
x = torch.tensor([1, 2, 3])
print("Vector * escalar:", x * 3)

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # (2,2)
b = torch.tensor([[1.0], [2.0]])            # (2,1) -> se estira a (2,2)
print("Matriz + columna:\n", a + b)

## CPU ↔ GPU: mover tensores de dispositivo

Esta es una diferencia práctica importante de PyTorch: **tú controlas explícitamente** en qué dispositivo vive cada tensor con `.to(device)`. Para que una operación funcione, **todos los tensores deben estar en el mismo dispositivo**. Es la fuente #1 de errores al usar GPU.

In [ ]:
x_cpu = torch.tensor([1.0, 2.0, 3.0])
x_dev = x_cpu.to(device)        # 'cuda' si hay GPU, si no se queda en 'cpu'
print("Tensor en:", x_dev.device)
# Para volver a NumPy desde GPU habría que hacer: x_dev.cpu().numpy()

## Estado entrenable: tensores con `requires_grad`

En TensorFlow el estado entrenable era `tf.Variable`. En PyTorch **cualquier tensor** puede volverse "entrenable" activando el atributo **`requires_grad=True`**. A partir de ese momento, PyTorch rastrea todas las operaciones sobre él para poder calcular gradientes.

Los pesos de una red (cuando usemos `torch.nn`, en el Módulo 3) tienen `requires_grad=True` automáticamente.

In [ ]:
w = torch.tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
print("¿rastrea gradientes?:", w.requires_grad)
print(w)

## Diferenciación automática: **autograd**

Llegamos al corazón del *Deep Learning*. El mecanismo de PyTorch se llama **autograd** y funciona así:

1. Marcas los tensores de interés con `requires_grad=True`.
2. Haces operaciones (el *forward*); PyTorch construye un **grafo dinámico** de todo lo que hiciste.
3. Llamas a **`.backward()`** sobre el resultado escalar (la *loss*): PyTorch recorre el grafo hacia atrás aplicando la regla de la cadena.
4. El gradiente de cada tensor queda guardado en su atributo **`.grad`**.

Compara con TensorFlow:

| Concepto | TensorFlow | PyTorch |
|---|---|---|
| Marcar entrenable | `tf.Variable` | `requires_grad=True` |
| Grabar operaciones | `with tf.GradientTape()` | automático si `requires_grad` |
| Calcular gradientes | `tape.gradient(y, x)` | `y.backward()` |
| Leer el gradiente | valor devuelto | `x.grad` |

### Ejemplo mínimo: derivada de $y = x^2$

En $x=3$, $\frac{dy}{dx}=2x=6$.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2
y.backward()             # calcula dy/dx y lo guarda en x.grad
print("dy/dx en x=3:", x.grad.item(), "(esperado: 6.0)")

### Gradiente respecto de varios parámetros

Simulamos una capa lineal $y = xW + b$ con pérdida cuadrática media, igual que hicimos en TensorFlow.

In [ ]:
torch.manual_seed(0)
w = torch.randn(3, 2, requires_grad=True)
b = torch.zeros(2, requires_grad=True)
x = torch.tensor([[1.0, 2.0, 3.0]])

y = x @ w + b
loss = (y**2).mean()
loss.backward()

print("Gradiente respecto de w (shape", w.grad.shape, "):\n", w.grad)
print("Gradiente respecto de b:", b.grad)

### Visualizar una derivada: la sigmoide y su pendiente

Para derivar respecto de un vector completo, usamos `torch.autograd.grad`, que devuelve el gradiente sin acumularlo en `.grad`.

In [ ]:
x = torch.linspace(-10, 10, 201, requires_grad=True)
y = torch.sigmoid(x)
# grad de una suma = gradiente de cada elemento (porque la sigmoide es elemento a elemento)
dy_dx, = torch.autograd.grad(y.sum(), x)

plt.figure(figsize=(7, 4))
plt.plot(x.detach(), y.detach(), label='σ(x)  (sigmoide)')
plt.plot(x.detach(), dy_dx, label="σ'(x)  (derivada)")
plt.legend(); plt.xlabel('x'); plt.grid(alpha=0.3)
plt.title("Una función y su derivada, con autograd")
plt.show()

> **`.detach()`** desconecta un tensor del grafo de autograd para poder graficarlo o convertirlo a NumPy. Es necesario cuando el tensor tiene `requires_grad=True`.

### Derivadas de orden superior

Con `create_graph=True` el propio cálculo del gradiente queda registrado, permitiendo derivar otra vez. Para $y=x^3$: primera derivada $3x^2$, segunda $6x$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**3
dy_dx, = torch.autograd.grad(y, x, create_graph=True)   # 3x^2 = 12
d2y_dx2, = torch.autograd.grad(dy_dx, x)                 # 6x = 12
print("primera derivada:", dy_dx.item(), "(esperado 12)")
print("segunda derivada:", d2y_dx2.item(), "(esperado 12)")

### Detalle práctico: los gradientes se **acumulan**

En PyTorch, `.grad` **suma** los gradientes de cada `.backward()` en lugar de reemplazarlos. Por eso, en un bucle de entrenamiento real, **hay que ponerlos a cero en cada paso** con `optimizer.zero_grad()` (o `x.grad.zero_()`). Es un tropiezo clásico de principiantes.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
for paso in range(3):
    y = x**2
    y.backward()
    print(f"paso {paso}: x.grad =", x.grad.item(), "(¡se va acumulando!)")
# La solución en entrenamiento real: x.grad.zero_() antes de cada backward()

### Reutilizar el grafo: `retain_graph`

En TensorFlow, una `GradientTape` normal se "consume" tras el primer `tape.gradient`, y para llamarla varias veces hace falta `persistent=True`. En PyTorch ocurre algo parecido: el grafo se libera tras el primer `backward`/`grad`. Si necesitas derivar **varias salidas** que comparten el mismo grafo, pasa `retain_graph=True`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2
z = x**3
dy_dx, = torch.autograd.grad(y, x, retain_graph=True)  # conserva el grafo
dz_dx, = torch.autograd.grad(z, x)                      # segunda derivación
print("dy/dx =", dy_dx.item(), "(2x  = 6)")
print("dz/dx =", dz_dx.item(), "(3x² = 27)")

### Congelar parámetros: `requires_grad = False`

Es el equivalente al `trainable=False` de TensorFlow. Un tensor con `requires_grad=False` no recibe gradiente porque PyTorch no lo rastrea; su `.grad` queda en `None`. Esta es la base del **transfer learning** (Módulo 6): congelar las capas preentrenadas y entrenar solo las nuevas.

In [ ]:
entrenable    = torch.tensor(1.0, requires_grad=True)
no_entrenable = torch.tensor(2.0, requires_grad=False)

y = entrenable * 2 + no_entrenable * 2
y.backward()
print("grad de entrenable   :", entrenable.grad)       # tensor(2.)
print("grad de no_entrenable:", no_entrenable.grad)    # None (congelado)

## Verificación de una solución analítica de una EDO con autograd

Un uso elegante de la diferenciación automática: comprobar si una función candidata resuelve una **ecuación diferencial ordinaria (EDO)**. Tomamos $y(t)=\cos(t)$ y verificamos que satisface $y'' + y = 0$. Calculamos $y'$ y $y''$ **con autograd** (no a mano) y evaluamos el **residuo** $y''+y$, que debe ser $\approx 0$.

Para derivar dos veces necesitamos `create_graph=True` en la primera derivación, de modo que el propio gradiente quede registrado y podamos volver a derivarlo. Esta idea es la semilla de las *Physics-Informed Neural Networks* (PINNs).

In [ ]:
t = torch.linspace(0.0, 2 * np.pi, 200, requires_grad=True)

y = torch.cos(t)
# primera derivada dy/dt (create_graph=True para poder volver a derivar)
dy_dt, = torch.autograd.grad(y, t, grad_outputs=torch.ones_like(t), create_graph=True)
# segunda derivada d²y/dt²
d2y_dt2, = torch.autograd.grad(dy_dt, t, grad_outputs=torch.ones_like(t))

residuo = d2y_dt2 + y            # debería ser ~0
t_np = t.detach().numpy()

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(t_np, y.detach(),       'b-', label=r'$y=\cos t$')
axs[0].plot(t_np, dy_dt.detach(),   'g-', label=r"$y'$")
axs[0].plot(t_np, d2y_dt2.detach(), 'm-', label=r"$y''$")
axs[0].legend(); axs[0].grid(alpha=0.3); axs[0].set_title("Función y sus derivadas")
axs[1].plot(t_np, residuo.detach(), 'r-')
axs[1].set_title(r"Residuo $y''+y$ (debe ser ~0)"); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Residuo máximo:", residuo.abs().max().item())

## MCMC: Metropolis-Hastings desde cero

Para cerrar, un ejemplo que **solo usa tensores** (sin autograd): un muestreador **Metropolis-Hastings** que genera muestras de una Normal estándar $\mathcal{N}(0,1)$. Es un buen ejercicio para ver el framework "en crudo".

1. Partimos de un estado $x_0$.
2. Proponemos $x' = x + \epsilon$, con $\epsilon \sim \mathcal{N}(0,\sigma^2)$ (propuesta simétrica).
3. Aceptamos con probabilidad $\alpha = \min\!\big(1,\, p(x')/p(x)\big)$.
4. Trabajamos en **escala logarítmica** para evitar desbordamientos: comparamos $\log u < \log\alpha$.

In [ ]:
torch.manual_seed(0)

def log_prob(x):                       # log-densidad Normal(0,1) sin normalizar
    return -0.5 * x**2

def mh_step(x, sigma=0.5):
    x_prop = x + torch.randn(()) * sigma           # propuesta simétrica
    log_alpha = log_prob(x_prop) - log_prob(x)
    log_u = torch.log(torch.rand(()))
    aceptar = bool(log_u < log_alpha)
    return (x_prop if aceptar else x), aceptar

def sample_chain(n=10000, burnin=5000, x0=5.0, sigma=0.5):
    x = torch.tensor(x0)
    muestras, aceptados = [], 0
    for i in range(n + burnin):
        x, acc = mh_step(x, sigma)
        if i >= burnin:
            muestras.append(x.item()); aceptados += acc
    return np.array(muestras), aceptados / n

samples, tasa = sample_chain()
print(f"Tasa de aceptación: {tasa:.3f}  (ideal ~0.23-0.45)")
print(f"Media: {samples.mean():.4f} (esp. 0) | Desv: {samples.std():.4f} (esp. 1)")

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(samples, linewidth=0.5); axs[0].set_title("Traceplot")
axs[0].set_xlabel("iteración"); axs[0].set_ylabel("x")
axs[1].hist(samples, bins=50, density=True, alpha=0.6, label="MCMC")
xg = np.linspace(-4, 4, 200)
axs[1].plot(xg, np.exp(-0.5 * xg**2) / np.sqrt(2 * np.pi), 'r-', label="Normal(0,1)")
axs[1].set_title("Distribución muestreada"); axs[1].legend()
plt.tight_layout(); plt.show()